# Bag of Words (BoW)

The **main idea behind Bag of Words (BoW)** when it first emerged (1950s–1970s in linguistics and information retrieval) was Represent a document as a collection (or “bag”) of its words, ignoring order, grammar, and syntax, but keeping track of which words appear and how often. **Zellig Harris (1954)** proposed the **distributional hypothesis**, i.e., "words that occur in similar contexts tend to have similar meanings." This set the stage for representing text through observable word distributions. **Gerard Salton (1960s–1980s)** in developing the **Vector Space Model** of Information Retrieval, he and colleagues operationalized this idea by mapping each document to a **vector of word frequencies**, enabling mathematical comparison between documents and queries.

### Intuitive reasoning at the time

1. **Content over order**: For tasks like **searching** or **classifying** documents, the *presence* of a word was more important than *where* it appeared in the text.
2. **Simplicity**: By discarding order, documents could be stored and processed as word histograms — a manageable abstraction given the computational limitations of the 1960s–70s.
3. **Similarity(overlap in vocabulary)**: Two texts about the same topic usually share many words; so comparing their word vectors made sense for retrieval or categorization.

So, the **main idea** was: *“Documents can be effectively represented and compared by treating them as bags of words, without needing to model syntax or order.”*



### Common Applications

- **Document classification**: spam filtering, sentiment analysis, topic tagging.  
- **Search & Information Retrieval**: ranking documents by similarity to a query.  
- **Clustering**: grouping similar documents without labels.  
- **Language identification**: character-level BoW can be very effective.  
- **As a baseline**: provides a strong, interpretable benchmark against which to compare modern neural methods.


### Mathematical Foundation
Let:
- $D = \{d_1, d_2, \ldots, d_N\}$ be a collection of $N$ documents.
- $V = \{w_1, w_2, \ldots, w_M\}$ be the vocabulary of $M$ unique words.

Then each document $d_i$ can be represented as a vector $\mathbf{x}_i \in \mathbb{R}^M$, where $x_{ij} = \text{count}(w_j, d_i)$ is frequency of word $j$ in document $i$.

The length of $\mathbf{x}_i$ is equal to the length of the vocabulary.

### Practical Example in Python

In [20]:
# Import necessary libraries
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Sample corpus
corpus = [
    "I love NLP, because NLP is very interesting",
    "NLP is fun",
    "I love machine learning and NLP"
]

# Create the BoW model
vectorizer = CountVectorizer()

X = vectorizer.fit_transform(corpus)

# Show the vocabulary and feature vectors
df = pd.DataFrame(X.toarray(),index=['Vector 1','Vector 2','Vector 3'], columns=vectorizer.get_feature_names_out())
df

,and,because,fun,interesting,is,learning,love,machine,nlp,very
Vector 1,0,1,0,1,1,0,1,0,2,1
Vector 2,0,0,1,0,1,0,0,0,1,0
Vector 3,1,0,0,0,0,1,1,1,1,0


### Interpretation

For example, in vector 1, the component corresponding to the word `NLP` is 2 which indicates that the word `NLP` appears twice in document 1, and the component corresponding to `learning` is 0, which means this word does not exist in document 1.

***Note**: The word `"I"` is missing in the output of CountVectorizer. This happens because by default, `CountVectorizer` removes English stop words (like "I", "the", "is", etc.). we can override the `token_pattern` to include `single-character` tokens like "I": `vectorizer = CountVectorizer(token_pattern=r"(?u)\b\w+\b")`

### Implement `from scratch`

- Step 1: Tokenization & Lowercase(preprocessing)

- Step 2: Build Vocabulary

- Step 3: Create BoW Vectors

In [21]:
import numpy as np
from collections import defaultdict

# Step 1: Tokenization & Lowercase

# this function converts the text to lower case, removes dotes and finally splits based on blank space
def tokenize(text): 
    return text.lower().replace('.', '').replace(',','').split()

tokenized_docs = [tokenize(doc) for doc in corpus]

# Step 2: Build Vocabulary
vocab = set()
# this loop adds all words in tokenized list to vocabulary(as a set)
for doc in tokenized_docs: vocab.update(doc)

# Optional: sorting thevocabulary
vocab = sorted(vocab)


# Step 3: Create BoW Vectors
bow_vectors = []

for doc in tokenized_docs:
    
    word_count = defaultdict(int)
    # Enumerates number of each vords in the document
    for word in doc:
        word_count[word] += 1
    # Generates vector
    vector = [word_count[word] for word in vocab]
    
    bow_vectors.append(vector)


# Show the vocabulary and feature vectors
df = pd.DataFrame(np.asarray(bow_vectors),index=['Vector 1','Vector 2','Vector 3'], columns=vocab)
df

,and,because,fun,i,interesting,is,learning,love,machine,nlp,very
Vector 1,0,1,0,1,1,1,0,1,0,2,1
Vector 2,0,0,1,0,0,1,0,0,0,1,0
Vector 3,1,0,0,1,0,0,1,1,1,1,0


### Real-World Example 
Sentiment Analysis of Short Customer Reviews using BoW.

We'll build a simple sentiment classifier (positive vs. negative) using **CountVectorizer** (Bag of Words) for feature extraction and **LogisticRegression** as a linear classifier

**Notes:**
- This is a *minimal* example with a tiny dataset to keep the notebook self-contained.

- In practice, you would use far more data and perform careful model evaluation.

- We add rich comments (#) throughout, as requested.

In [22]:
from typing import List, Tuple

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# -----------------------------
# 1) Tiny, self-contained dataset
# -----------------------------
# We'll simulate short customer reviews. Labels: 1 = positive, 0 = negative.
data: List[Tuple[str, int]] = [
    ("Absolutely loved the product, great quality and fast shipping", 1), ("Terrible experience, item arrived broken and support ignored me", 0),
    ("Works as expected, very satisfied with my purchase", 1), ("Do not buy this, waste of money and time", 0),
    ("The packaging was neat and the item works perfectly", 1), ("Received the wrong color and the material feels cheap", 0),
    ("Customer service was helpful and resolved my issue quickly", 1), ("Late delivery, poor quality, not recommended", 0),
    ("Five stars! Exceeded my expectations in every way", 1), ("Battery died after one day, extremely disappointed", 0),
    ("Setup was easy and instructions were clear", 1), ("The size chart is misleading and the fit is terrible", 0),
    ("I will buy again, excellent value", 1), ("Refund process was difficult and slow", 0), ("Super comfortable and looks great", 1),
    ("Feels flimsy and stopped working after a week", 0), ("Amazing sound quality for the price", 1),("Shipping took forever and the box was damaged", 0),
    ("Exactly what I needed, highly recommend", 1),("Worst purchase this year", 0),
]

df = pd.DataFrame(data, columns=["text", "label"])

# -----------------------------
# 2) Train/Validation split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=42, stratify=df["label"]
)

# -----------------------------
# 3) Build a BoW pipeline
# -----------------------------
# CountVectorizer converts text to a sparse matrix of token counts.
# We keep it simple:
#   - lowercase=True (default)
#   - stop_words="english" to drop common words (like 'the', 'and') that rarely help
#   - ngram_range=(1,1) (unigrams only) to stay faithful to basic BoW
pipe = Pipeline([ ("bow", CountVectorizer(stop_words="english")),("clf", LogisticRegression(max_iter=1000))])

# -----------------------------
# 4) Fit the model
# -----------------------------
pipe.fit(X_train, y_train)

# -----------------------------
# 5) Evaluate
# -----------------------------
pred = pipe.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("\nClassification report:\n", classification_report(y_test, pred))

# -----------------------------
# 6) Inspect important features
# -----------------------------
# We'll examine the learned coefficients to see which words push predictions positive/negative.
#   - Get the vocabulary and classifier weights
vectorizer: CountVectorizer = pipe.named_steps["bow"]
clf: LogisticRegression = pipe.named_steps["clf"]

feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]  # binary classification -> one coefficient vector


Accuracy: 0.333

Classification report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.40      0.67      0.50         3

    accuracy                           0.33         6
   macro avg       0.20      0.33      0.25         6
weighted avg       0.20      0.33      0.25         6



### Top 10 word ranking

Top positive and negative indicative words

In [23]:
# Top positive / negative indicative words
topn = 10
pos_idx = np.argsort(coefs)[-topn:][::-1]
neg_idx = np.argsort(coefs)[:topn]

top_pos = list(zip(feature_names[pos_idx], coefs[pos_idx]))
top_neg = list(zip(feature_names[neg_idx], coefs[neg_idx]))

print("\nTop positive words:")
for w, c in top_pos: print(f"{w:>15s}  {c: .3f}")

print("\nTop negative words:")
for w, c in top_neg: print(f"{w:>15s}  {c: .3f}")


Top positive words:
          great   0.419
        quality   0.419
      satisfied   0.295
          works   0.295
       expected   0.295
          setup   0.258
   expectations   0.258
       exceeded   0.258
           easy   0.258
          clear   0.258

Top negative words:
          feels  -0.419
       terrible  -0.405
          worst  -0.336
           year  -0.336
          money  -0.264
            buy  -0.264
          waste  -0.264
           time  -0.264
        forever  -0.256
           took  -0.256


### Test the model

using the model on new and unseen data to predict the labels

In [24]:
# -----------------------------
# 7) Use the model on new data
# -----------------------------
new_texts = [ "The headphones are fantastic and comfortable", "Support was rude and I want a refund"]

preds = pipe.predict(new_texts)

for txt, p in zip(new_texts, preds):
    sentiment = "positive" if p == 1 else "negative"
    print(f"{sentiment}: \"{txt}\"")

positive: "The headphones are fantastic and comfortable"
negative: "Support was rude and I want a refund"


### Code Commentary (What Each Part Does)

- **Dataset**: For reproducibility and offline use, we create a *tiny* labeled dataset of short customer reviews. In practice, you'd use a much larger, real dataset.  
- **Split**: We split into train/test to simulate evaluation on unseen data.  
- **CountVectorizer (BoW)**: Maps each document to a sparse vector of word counts. Using `stop_words="english"` is a simple heuristic to remove common function words.  
- **Classifier**: A linear model (`LogisticRegression`) is fast and works well with BoW features.  
- **Evaluation**: Accuracy + a classification report (precision/recall/F1).  
- **Feature inspection**: We look at the highest/lowest coefficients to see which words sway the classifier toward positive vs. negative predictions.  
- **Prediction**: We apply the trained pipeline to a couple of new, unseen texts.


### Advantages and Disadvantages
**Advantages**
- Simple to implement and interpret
- Works well with simpler models (e.g., Naive Bayes)

**Disadvantages**
- BoW ignores word order and semantics; modern embedding methods (e.g., word2vec, BERT) capture richer context. Nonetheless, BoW remains a strong, interpretable baseline.
- High dimensionality with large vocabulary
- Doesn’t handle synonyms or polysemy well

## References (Books & Peer-Reviewed Articles)

- Harris, Z. S. (1954). **Distributional structure**. *Word*, 10(2–3), 146–162.  
- Salton, G., Wong, A., & Yang, C.-S. (1975). **A vector space model for automatic indexing**. *Communications of the ACM*, 18(11), 613–620.  
- Salton, G., & McGill, M. J. (1983). **Introduction to Modern Information Retrieval**. McGraw-Hill.  
- Manning, C. D., Raghavan, P., & Schütze, H. (2008). **Introduction to Information Retrieval**. Cambridge University Press.  
- Joachims, T. (1998). **Text Categorization with Support Vector Machines: Learning with Many Relevant Features**. *ECML*, 137–142.
